# Per-condition statistical comparisons of compaction and actin metrics

Computes paired-sample statistics and produces publication figures comparing
two experimental conditions (e.g. Control vs. DeAct) across compaction- and
actin-related cell-level measurements. Single-cell measurements and
biological-replicate means are loaded from a single combined dataframe; the
notebook filters cells flagged for omission, computes per-replicate means,
runs a paired t-test (with a Shapiro-Wilk normality check on the
between-condition difference) for each measurement, and saves an SVG figure
plus a combined stats CSV. A second comparison contrasts mean actin
intensity in CAAX-positive versus compacted regions within control cells.

A provenance manifest written alongside the figures records the input file
hash, all configuration parameters, the list of output files, the git
commit, and a UTC timestamp. The final cell exports this notebook to
Markdown next to the figures.

In [ ]:
# --- Built-in modules ---

# --- Core scientific stack ---

# --- Local modules ---

# --- Notebook display ---
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Config

All run-specific paths and parameters are declared here. Style parameters
(rc settings, palettes, figure sizes) are imported from
``microscopy_analysis.d04_plot_data.plot_settings``; nothing below this
cell needs editing.

- `DF_PATH`: combined per-cell measurement CSV (one row per cell).
- `NOTEBOOK_NAME`: filename of this notebook, used by the manifest and
  Markdown-export cells.
- `CONDITION_COL`, `CONDITION_ORDER`: condition column and the two
  condition values to compare, in left-to-right plot order.
- `CMP_YCOLS`, `ACTIN_YCOLS`: measurement columns aggregated and plotted
  for the compaction and actin analyses, respectively.
- `GROUPBY_COLS`: columns identifying a single biological replicate.

In [ ]:
# Path to combined per-cell measurement CSV.
DF_PATH = Path('/Users/kwu2/Library/CloudStorage/GoogleDrive-kwu2@stanford.edu/My Drive/Lab/OL_compaction/Experiments_compiled/actin_pharmacological/data/combined_analysis.csv')

# This notebook's filename, used for manifest provenance and Markdown export.
NOTEBOOK_NAME = '10_plot_exclusion_analysis_multirep_v2.ipynb'

# Condition column and the pair of values to compare (in plot order).
CONDITION_COL = 'condition'
CONDITION_ORDER = ['Control', 'DeAct']

# Columns identifying a single biological replicate.
GROUPBY_COLS = ['condition', 'experiment']

# Compaction metrics aggregated and plotted per condition.
CMP_YCOLS = [
    'cell area',
    'compacted area',
    'CAAX-positive area',
    '% compaction',
    'mean caax int (cell)',
    'caax integrated density',
    'mean caax int (compacted)',
    'mean caax int (caax)',
]

# Actin metrics aggregated and plotted per condition.
ACTIN_YCOLS = [
    'mean actin int (cell)',
    'actin integrated density',
    'mean actin int (compacted)',
    'mean actin int (caax)',
]

## Load and filter data

Cells flagged in the `omit` column are dropped from the compaction
analysis; cells flagged in the `actin omit` column are dropped from the
actin analysis. The number of cells dropped at each step is printed for
the run record.

In [ ]:
df = pd.read_csv(DF_PATH)
df.head()

In [ ]:
# Drop cells flagged for omission from the compaction analysis.
cmp_df = df
if 'omit' in df.columns:
    num_cells_prefilter = len(cmp_df)
    cmp_df = cmp_df[cmp_df['omit'] != 'Y']
    num_cells_postfilter = len(cmp_df)
    print(f'num cells omitted: {num_cells_prefilter - num_cells_postfilter}')
else:
    print('No column indicating cells to omit.')

# Drop cells flagged for omission from the actin analysis.
actin_df = df
if 'actin omit' in df.columns:
    num_cells_prefilter = len(actin_df)
    actin_df = df[df['actin omit'] != 'Y']
    num_cells_postfilter = len(actin_df)
    print(f'num cells omitted for actin analysis: {num_cells_prefilter - num_cells_postfilter}')
else:
    print('No column indicating cells to omit for actin analysis.')

## Output directories and per-experiment cell counts

Output tables and figures are written next to the input CSV. Per-condition,
per-experiment cell counts are saved separately for the compaction and
actin analyses to document the sample size underlying each statistical
test.

In [ ]:
tables_dirpath = DF_PATH.parent
graphs_dirpath = DF_PATH.parent / 'graphs'
graphs_dirpath.mkdir(parents=True, exist_ok=True)

In [ ]:
# Per-condition, per-experiment cell counts for the compaction analysis.
cmp_counts = cmp_df.groupby(['condition', 'experiment'])['condition'].count()
cmp_counts_df_path = tables_dirpath / 'cmp_data_counts.csv'
cmp_counts.to_csv(cmp_counts_df_path, index=True)

# Per-condition, per-experiment cell counts for the actin analysis.
actin_counts = actin_df.groupby(['condition', 'experiment'])['condition'].count()
actin_counts_df_path = tables_dirpath / 'actin_data_counts.csv'
actin_counts.to_csv(actin_counts_df_path, index=True)

actin_counts

In [ ]:
# Biological-replicate means for compaction metrics.
agg_cols = {y: 'mean' for y in CMP_YCOLS}
cmp_biorepavg_df = cmp_df.groupby(GROUPBY_COLS, as_index=False).agg(agg_cols)
cmp_biorepavg_df_path = tables_dirpath / 'cmp_biorep_avgs.csv'
utils.safe_save_csv(cmp_biorepavg_df, cmp_biorepavg_df_path)

# Biological-replicate means for actin metrics.
agg_cols = {y: 'mean' for y in ACTIN_YCOLS}
actin_biorepavg_df = actin_df.groupby(GROUPBY_COLS, as_index=False).agg(agg_cols)
actin_biorepavg_df_path = tables_dirpath / 'actin_biorep_avgs.csv'
utils.safe_save_csv(actin_biorepavg_df, actin_biorepavg_df_path)

cmp_biorepavg_df

## Per-condition statistics and figures

For each measurement column, biological-replicate means are compared
between the two conditions with a paired t-test, paired by `experiment`. A
Shapiro-Wilk test on the between-condition difference reports normality of
the paired differences, which is the relevant assumption of the paired
t-test. Each comparison produces an SVG figure showing single-cell points,
biological-replicate means colored by condition, and the cross-replicate
mean ± standard error, annotated with the t-test p-value.

In [ ]:
def get_stats_and_graph(
    x,
    y,
    x_order,
    df,
    avg_df,
    stats_df,
    graphs_dirpath,
    graphname=None,
    figsize=(2.5, 3.25),
    palette=ctrldeact_palette,
):
    """Run a paired t-test between two conditions and save the comparison figure.

    A row is appended to ``stats_df`` recording the comparison value, the
    two condition labels, per-group sample sizes, the Shapiro-Wilk
    normality statistic on the paired difference, and the paired t-test
    statistic and p-value. Single-cell points (from ``df``) and
    biological-replicate means (from ``avg_df``, colored by ``palette``)
    are plotted, with the cross-replicate mean ± standard error overlaid
    in black. The p-value is annotated above the means with starbars and
    the figure is saved as SVG to ``graphs_dirpath``.

    Parameters
    ----------
    x : str
        Column in ``df`` and ``avg_df`` holding the categorical condition.
    y : str
        Measurement column to compare.
    x_order : list of str
        The two condition labels, in left-to-right plot order.
    df : pandas.DataFrame
        Per-cell measurements; plotted as small swarm points.
    avg_df : pandas.DataFrame
        Biological-replicate means; one row per condition x replicate.
    stats_df : pandas.DataFrame
        Accumulating stats table; a new row is appended and the updated
        frame is returned.
    graphs_dirpath : pathlib.Path
        Directory in which the SVG figure is saved.
    graphname : str, optional
        Output SVG filename. Defaults to a sanitized form of ``y``.
    figsize : tuple of float, optional
        Figure size in inches.
    palette : list of str, optional
        Two-color palette for the biological-replicate points.

    Returns
    -------
    pandas.DataFrame
        ``stats_df`` with one new row appended.
    """
    group1_name, group2_name = x_order[0], x_order[1]

    idx = len(stats_df)
    stats_df.loc[idx, 'comparison value'] = y
    stats_df.loc[idx, 'comparison group 1'] = group1_name
    stats_df.loc[idx, 'comparison group 2'] = group2_name
    group1 = avg_df.loc[avg_df['condition'] == group1_name, y].values
    group2 = avg_df.loc[avg_df['condition'] == group2_name, y].values
    stats_df.loc[idx, 'group 1 n'] = len(group1)
    stats_df.loc[idx, 'group 2 n'] = len(group2)

    # Shapiro-Wilk test on the paired difference (the assumption of the
    # paired t-test is normality of the differences, not the raw values).
    diff = group2 - group1
    normality_stat, normality_pvalue = stats.shapiro(diff)
    stats_df.loc[idx, 'Shapiro-Wilk normality stat'] = normality_stat
    stats_df.loc[idx, 'Shapiro-Wilk normality p-value'] = normality_pvalue

    # Paired t-test (paired by experiment via the row order in avg_df).
    stat, pvalue = stats.ttest_rel(group1, group2)
    stats_df.loc[idx, 'paired t-test stat'] = stat
    stats_df.loc[idx, 'paired t-test p-value'] = pvalue

    # Build figure: single-cell points, replicate means, replicate mean +/- SE.
    fig, ax = plt.subplots(figsize=figsize, dpi=150)
    sns.swarmplot(
        x=x, y=y, data=df, order=x_order, size=2.5, ax=ax,
        color=smallpts_fillcolor, edgecolor=smallpts_edgecolor, linewidth=0.2,
    )
    sns.swarmplot(
        x=x, y=y, order=x_order, data=avg_df, hue='condition',
        palette=palette, legend=False, size=7, ax=ax,
    )
    sns.pointplot(
        data=avg_df, x=x, y=y, order=x_order, linestyle='', errorbar='se',
        marker='_', markersize=25, markeredgewidth=2.5, zorder=3, color='k',
        capsize=0.1, err_kws={'linewidth': 1}, ax=ax,
    )
    starbars.draw_annotation(
        [(group1_name, group2_name, pvalue)],
        line_width=axes_linewidth, color='k', ax=ax,
    )
    sns.despine()
    plt.ylim(0)
    plt.xlim(-0.7, 1.7)
    ax.xaxis.label.set_visible(False)
    ax.tick_params(direction='out', width=axes_linewidth, labelsize=xtick_fontsize)
    plt.show()

    if graphname is None:
        graphname = (
            y.replace('(', '').replace(')', '').replace(' ', '_').replace('%', 'perc')
            + '.svg'
        )
    fig.savefig(graphs_dirpath / graphname, format='svg', bbox_inches='tight')

    return stats_df

In [ ]:
stats_df = pd.DataFrame()

# Compaction comparisons. CAAX intensity panels use a slightly wider
# figure so that long axis labels render without overlap.
for y in CMP_YCOLS:
    figsize = (3, 3.25) if 'caax' in y else (2.5, 3.25)
    stats_df = get_stats_and_graph(
        CONDITION_COL, y, CONDITION_ORDER,
        df=cmp_df, avg_df=cmp_biorepavg_df, stats_df=stats_df,
        graphs_dirpath=graphs_dirpath, figsize=figsize,
    )

# Actin comparisons.
for y in ACTIN_YCOLS:
    stats_df = get_stats_and_graph(
        CONDITION_COL, y, CONDITION_ORDER,
        df=actin_df, avg_df=actin_biorepavg_df, stats_df=stats_df,
        graphs_dirpath=graphs_dirpath, figsize=(2.5, 3.25),
    )

## Actin intensity in CAAX-positive vs. compacted regions (control cells)

Within control cells only, mean actin intensity is compared between the
CAAX-positive (non-compacted) region and the compacted region. The
two-region structure is plotted with both points in the control color so
the visual highlights the within-cell regional contrast rather than a
between-condition difference.

In [ ]:
# Restructure to long form so 'condition' encodes the within-cell region
# (caax / compacted / cell) for the control subset.
actin_df_ctrl = actin_df[actin_df['condition'] == 'Control']
actin_dfm_ctrl = actin_df_ctrl.melt(
    id_vars=['UID', 'condition'],
    value_vars=['mean actin int (caax)', 'mean actin int (compacted)', 'mean actin int (cell)'],
    value_name='mean actin int',
)

actin_avg_df_ctrl = actin_biorepavg_df[actin_biorepavg_df['condition'] == 'Control']
actin_avg_dfm_ctrl = actin_avg_df_ctrl.melt(
    id_vars=['experiment', 'condition'],
    value_vars=['mean actin int (caax)', 'mean actin int (compacted)', 'mean actin int (cell)'],
    value_name='mean actin int',
)

regions = ['cell', 'caax', 'compacted']
for reg in regions:
    actin_dfm_ctrl.loc[actin_dfm_ctrl['variable'].str.contains(reg), 'condition'] = reg
    actin_avg_dfm_ctrl.loc[actin_avg_dfm_ctrl['variable'].str.contains(reg), 'condition'] = reg

In [ ]:
# Single-color palette: both regions plotted in the control color so the
# figure reads as a within-cell regional comparison, not a
# between-condition contrast.
ctrl_palette = [ctrldeact_palette[0], ctrldeact_palette[0]]
stats_df = get_stats_and_graph(
    'condition', 'mean actin int', ['caax', 'compacted'],
    df=actin_dfm_ctrl, avg_df=actin_avg_dfm_ctrl, stats_df=stats_df,
    graphs_dirpath=graphs_dirpath, palette=ctrl_palette,
)

## Save combined statistics

Duplicate rows (a comparison appearing more than once across the loops
above) are collapsed before writing the combined stats table.

In [ ]:
stats_df = stats_df.drop_duplicates()
stats_df_path = tables_dirpath / 'stats.csv'
utils.safe_save_csv(stats_df, stats_df_path)

## Provenance manifest

Writes a JSON manifest next to the figures that records, for this run:
the input CSV path and SHA-256 hash; every configuration parameter
declared in the config cell; the list of figure and table files
generated; the git commit of this analysis repository; and a UTC
timestamp. The manifest is intended to make every published figure
traceable back to a specific input file, code revision, and parameter
set.

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()


def git_commit(repo_dir):
    try:
        return subprocess.check_output(
            ['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'],
            stderr=subprocess.DEVNULL,
        ).decode().strip()
    except Exception:
        return None


# Output files generated by this notebook (figures + tables).
output_files = sorted(
    [str(p) for p in graphs_dirpath.glob('*.svg')]
    + [
        str(cmp_counts_df_path),
        str(actin_counts_df_path),
        str(cmp_biorepavg_df_path),
        str(actin_biorepavg_df_path),
        str(stats_df_path),
    ]
)

manifest = {
    'notebook': NOTEBOOK_NAME,
    'input': {
        'path': str(DF_PATH),
        'sha256': file_sha256(DF_PATH),
    },
    'config': {
        'CONDITION_COL': CONDITION_COL,
        'CONDITION_ORDER': CONDITION_ORDER,
        'GROUPBY_COLS': GROUPBY_COLS,
        'CMP_YCOLS': CMP_YCOLS,
        'ACTIN_YCOLS': ACTIN_YCOLS,
    },
    'outputs': output_files,
    'git_commit': git_commit(Path.cwd()),
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}

manifest_path = graphs_dirpath / f'{Path(NOTEBOOK_NAME).stem}.manifest.json'
with open(manifest_path, 'w') as fh:
    json.dump(manifest, fh, indent=2)

print(f'Wrote {manifest_path}')

## Export this notebook to Markdown

Renders the executed notebook (code, prose, and inline figure references)
to a Markdown file in `graphs_dirpath`, alongside the SVG figures and the
manifest, so the figures and the analysis steps that produced them are
archived together.

In [ ]:
notebook_path = Path(NOTEBOOK_NAME).resolve()
subprocess.run(
    [
        'jupyter', 'nbconvert',
        '--to', 'markdown',
        str(notebook_path),
        '--output-dir', str(graphs_dirpath),
    ],
    check=True,
)